# 08 Evaluation

This notebook consolidates model prediction outputs into discrimination, calibration, confusion-matrix, and clinical-utility artifacts. It is designed to support both baseline and multimodal models through a common prediction-table format.

## Evaluation scope

- AUROC and AUPRC
- Precision, recall, F1, sensitivity, specificity
- Brier score and calibration tables
- ROC and PR curve points
- Lead-time summaries by horizon
- Reusable CSV artifacts for figure generation

In [1]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [2]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn

In [3]:
import pandas as pd

from src.evaluation.artifacts import build_lead_time_table, summarize_predictions
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [4]:
config['evaluation']

{'default_threshold': 0.5, 'calibration_bins': 10}

## Collect prediction tables

This notebook evaluates three families of prediction artifacts through the same interface:

- structured baselines from Notebook 06
- tabular custom multimodal models stored in `06_baseline_models`
- deep multimodal fusion models from Notebook 07

In [5]:
baseline_dir = paths['processed_data_dir'] / '06_baseline_models'
multimodal_dir = paths['processed_data_dir'] / '07_multimodal_models'

baseline_prediction_files = sorted(baseline_dir.glob('*_test_predictions.csv'))
multimodal_prediction_files = sorted(multimodal_dir.glob('*_test_predictions.csv'))
prediction_files = sorted([*baseline_prediction_files, *multimodal_prediction_files])

assert prediction_files, 'No prediction files found. Run 06_baseline_models or 07_multimodal_models first.'

pd.DataFrame({
    'source_stage': [path.parent.name for path in prediction_files],
    'prediction_file': [path.name for path in prediction_files],
}).head(20)

,source_stage,prediction_file
0,06_baseline_models,horizon_12h_logistic_regression_test_predictio...
1,06_baseline_models,horizon_12h_random_forest_test_predictions.csv
2,06_baseline_models,horizon_12h_xgboost_test_predictions.csv
3,06_baseline_models,horizon_24h_logistic_regression_test_predictio...
4,06_baseline_models,horizon_24h_random_forest_test_predictions.csv
5,06_baseline_models,horizon_24h_stacked_xgboost_notes_test_predict...
6,06_baseline_models,horizon_24h_xgboost_test_predictions.csv
7,06_baseline_models,horizon_24h_xgboost_text_augmented_test_predic...
8,06_baseline_models,horizon_6h_logistic_regression_test_prediction...
9,06_baseline_models,horizon_6h_random_forest_test_predictions.csv


## Evaluate each prediction file

In [6]:
DEEP_MULTIMODAL_MODELS = {
    'early_fusion',
    'late_fusion',
    'gated_fusion',
    'cross_modal_attention',
}
TABULAR_MULTIMODAL_MODELS = {
    'xgboost_text_augmented',
    'stacked_xgboost_notes',
}
STRUCTURED_BASELINE_MODELS = {
    'logistic_regression',
    'random_forest',
    'xgboost',
}

def infer_model_family(model_name: str, source_stage: str) -> str:
    if model_name in TABULAR_MULTIMODAL_MODELS:
        return 'custom_tabular_multimodal'
    if model_name in DEEP_MULTIMODAL_MODELS:
        return 'custom_deep_multimodal'
    if model_name in STRUCTURED_BASELINE_MODELS:
        return 'structured_baseline'
    if source_stage == '07_multimodal_models':
        return 'custom_deep_multimodal'
    if source_stage == '06_baseline_models':
        return 'structured_baseline'
    return 'other'

metric_rows = []
artifact_bundle = {}

for path in prediction_files:
    predictions_df = pd.read_csv(path)
    model_name = predictions_df['model_name'].iloc[0] if 'model_name' in predictions_df.columns else path.stem
    dataset_name = predictions_df['dataset_name'].iloc[0] if 'dataset_name' in predictions_df.columns else 'unknown_dataset'
    source_stage = path.parent.name
    model_family = infer_model_family(model_name=model_name, source_stage=source_stage)
    horizon_hours = int(dataset_name.replace('horizon_', '').replace('h', '')) if dataset_name.startswith('horizon_') else -1

    metrics, curves = summarize_predictions(
        predictions_df=predictions_df,
        threshold=config['evaluation']['default_threshold'],
        calibration_bins=config['evaluation']['calibration_bins'],
    )
    metric_rows.append({
        'dataset_name': dataset_name,
        'model_name': model_name,
        'model_family': model_family,
        'source_stage': source_stage,
        'prediction_file': path.name,
        'split': 'test',
        **metrics,
    })

    lead_time_df = build_lead_time_table(predictions_df, horizon_hours=horizon_hours)
    artifact_bundle[f'{dataset_name}_{model_name}_roc_curve'] = curves['roc_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_pr_curve'] = curves['pr_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_calibration'] = curves['calibration']
    artifact_bundle[f'{dataset_name}_{model_name}_lead_time'] = lead_time_df

evaluation_df = pd.DataFrame(metric_rows).sort_values(
    ['dataset_name', 'model_family', 'auprc', 'model_name'],
    ascending=[True, True, False, True],
).reset_index(drop=True)
evaluation_df

,dataset_name,model_name,model_family,source_stage,prediction_file,split,auroc,auprc,accuracy,precision,recall,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,decision_threshold
0,horizon_12h,cross_modal_attention,custom_deep_multimodal,07_multimodal_models,horizon_12h_cross_modal_attention_test_predict...,test,0.670186,0.100267,0.877510,0.130068,0.322176,0.185319,0.040907,0.902610,0.322176,77,515,4773,162,0.052258
1,horizon_12h,late_fusion,custom_deep_multimodal,07_multimodal_models,horizon_12h_late_fusion_test_predictions.csv,test,0.660434,0.091342,0.891261,0.126033,0.255230,0.168741,0.043496,0.920008,0.255230,61,423,4865,178,0.120319
2,horizon_12h,gated_fusion,custom_deep_multimodal,07_multimodal_models,horizon_12h_gated_fusion_test_predictions.csv,test,0.650558,0.090082,0.799349,0.077670,0.334728,0.126084,0.052310,0.820348,0.334728,80,950,4338,159,0.170048
3,horizon_12h,early_fusion,custom_deep_multimodal,07_multimodal_models,horizon_12h_early_fusion_test_predictions.csv,test,0.636393,0.082739,0.760268,0.081664,0.443515,0.137931,0.041388,0.774584,0.443515,106,1192,4096,133,0.020540
4,horizon_12h,random_forest,structured_baseline,06_baseline_models,horizon_12h_random_forest_test_predictions.csv,test,0.998746,0.980104,0.998010,0.983051,0.970711,0.976842,0.006747,0.999244,0.970711,232,4,5284,7,0.500000
5,horizon_12h,xgboost,structured_baseline,06_baseline_models,horizon_12h_xgboost_test_predictions.csv,test,0.998611,0.961338,0.995296,0.931174,0.962343,0.946502,0.004433,0.996785,0.962343,230,17,5271,9,0.500000
6,horizon_12h,logistic_regression,structured_baseline,06_baseline_models,horizon_12h_logistic_regression_test_predictio...,test,0.853810,0.265201,0.799530,0.152678,0.799163,0.256376,0.151014,0.799546,0.799163,191,1060,4228,48,0.500000
7,horizon_24h,late_fusion,custom_deep_multimodal,07_multimodal_models,horizon_24h_late_fusion_test_predictions.csv,test,0.743269,0.168528,0.920110,0.195833,0.268571,0.226506,0.043051,0.949779,0.268571,47,193,3650,128,0.005552
8,horizon_24h,cross_modal_attention,custom_deep_multimodal,07_multimodal_models,horizon_24h_cross_modal_attention_test_predict...,test,0.744746,0.164070,0.931807,0.220339,0.222857,0.221591,0.043235,0.964091,0.222857,39,138,3705,136,0.003785
9,horizon_24h,early_fusion,custom_deep_multimodal,07_multimodal_models,horizon_24h_early_fusion_test_predictions.csv,test,0.725875,0.136139,0.885764,0.132124,0.291429,0.181818,0.043000,0.912829,0.291429,51,335,3508,124,0.005196


## Inspect strongest test results

In [7]:
summary_columns = [
    'dataset_name',
    'model_family',
    'model_name',
    'source_stage',
    'auroc',
    'auprc',
    'accuracy',
    'precision',
    'recall',
    'f1',
    'decision_threshold',
]

top_overall_df = (
    evaluation_df.loc[:, summary_columns]
    .sort_values(['dataset_name', 'auprc'], ascending=[True, False])
    .reset_index(drop=True)
)
custom_multimodal_df = (
    evaluation_df.loc[evaluation_df['model_family'].str.startswith('custom_'), summary_columns]
    .sort_values(['dataset_name', 'auprc'], ascending=[True, False])
    .reset_index(drop=True)
)

display(top_overall_df.head(15))
display(custom_multimodal_df)

,dataset_name,model_family,model_name,source_stage,auroc,auprc,accuracy,precision,recall,f1,decision_threshold
0,horizon_12h,structured_baseline,random_forest,06_baseline_models,0.998746,0.980104,0.998010,0.983051,0.970711,0.976842,0.500000
1,horizon_12h,structured_baseline,xgboost,06_baseline_models,0.998611,0.961338,0.995296,0.931174,0.962343,0.946502,0.500000
2,horizon_12h,structured_baseline,logistic_regression,06_baseline_models,0.853810,0.265201,0.799530,0.152678,0.799163,0.256376,0.500000
3,horizon_12h,custom_deep_multimodal,cross_modal_attention,07_multimodal_models,0.670186,0.100267,0.877510,0.130068,0.322176,0.185319,0.052258
4,horizon_12h,custom_deep_multimodal,late_fusion,07_multimodal_models,0.660434,0.091342,0.891261,0.126033,0.255230,0.168741,0.120319
5,horizon_12h,custom_deep_multimodal,gated_fusion,07_multimodal_models,0.650558,0.090082,0.799349,0.077670,0.334728,0.126084,0.170048
6,horizon_12h,custom_deep_multimodal,early_fusion,07_multimodal_models,0.636393,0.082739,0.760268,0.081664,0.443515,0.137931,0.020540
7,horizon_24h,custom_tabular_multimodal,xgboost_text_augmented,06_baseline_models,0.999359,0.982061,0.996267,0.934783,0.982857,0.958217,0.400000
8,horizon_24h,structured_baseline,random_forest,06_baseline_models,0.998785,0.975328,0.998756,0.982955,0.988571,0.985755,0.500000
9,horizon_24h,structured_baseline,xgboost,06_baseline_models,0.998962,0.965150,0.995769,0.929348,0.977143,0.952646,0.500000


,dataset_name,model_family,model_name,source_stage,auroc,auprc,accuracy,precision,recall,f1,decision_threshold
0,horizon_12h,custom_deep_multimodal,cross_modal_attention,07_multimodal_models,0.670186,0.100267,0.877510,0.130068,0.322176,0.185319,0.052258
1,horizon_12h,custom_deep_multimodal,late_fusion,07_multimodal_models,0.660434,0.091342,0.891261,0.126033,0.255230,0.168741,0.120319
2,horizon_12h,custom_deep_multimodal,gated_fusion,07_multimodal_models,0.650558,0.090082,0.799349,0.077670,0.334728,0.126084,0.170048
3,horizon_12h,custom_deep_multimodal,early_fusion,07_multimodal_models,0.636393,0.082739,0.760268,0.081664,0.443515,0.137931,0.020540
4,horizon_24h,custom_tabular_multimodal,xgboost_text_augmented,06_baseline_models,0.999359,0.982061,0.996267,0.934783,0.982857,0.958217,0.400000
5,horizon_24h,custom_tabular_multimodal,stacked_xgboost_notes,06_baseline_models,0.996619,0.929053,0.995769,0.920213,0.988571,0.953168,0.500000
6,horizon_24h,custom_deep_multimodal,late_fusion,07_multimodal_models,0.743269,0.168528,0.920110,0.195833,0.268571,0.226506,0.005552
7,horizon_24h,custom_deep_multimodal,cross_modal_attention,07_multimodal_models,0.744746,0.164070,0.931807,0.220339,0.222857,0.221591,0.003785
8,horizon_24h,custom_deep_multimodal,early_fusion,07_multimodal_models,0.725875,0.136139,0.885764,0.132124,0.291429,0.181818,0.005196
9,horizon_24h,custom_deep_multimodal,gated_fusion,07_multimodal_models,0.636047,0.074712,0.928820,0.088889,0.068571,0.077419,0.011135


## Save evaluation artifacts

In [8]:
output_dir = paths['processed_data_dir'] / '08_evaluation'
artifact_bundle['evaluation_summary'] = evaluation_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'horizon_12h_logistic_regression_roc_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/08_evaluation/horizon_12h_logistic_regression_roc_curve.csv',
 'horizon_12h_logistic_regression_pr_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/08_evaluation/horizon_12h_logistic_regression_pr_curve.csv',
 'horizon_12h_logistic_regression_calibration': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/08_evaluation/horizon_12h_logistic_regression_calibration.csv',
 'horizon_12h_logistic_regression_lead_time': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/08_evaluation/horizon_12h_logistic_regression_lead_time.csv',
 'horizon_12h_random_forest_roc_curve': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/08_evaluation/horizon_12h_random_forest_roc_curve.csv',
 'horizon_12h_random_forest_pr_curve': '/home/sra/shankari/Multimodal-Sepsis-Predicti

In [9]:
manifest_path = paths['manifests_dir'] / '08_evaluation_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='08_evaluation',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'prediction_files_evaluated': [str(path) for path in prediction_files],
        'num_models_evaluated': int(evaluation_df['model_name'].nunique()) if not evaluation_df.empty else 0,
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/08_evaluation_manifest.json')

## Review checklist

Before ablation analysis, review:

- Which model wins on AUROC versus AUPRC?
- Are calibration tables noticeably misaligned?
- Does performance degrade as horizon length increases?
- Are sensitivity and specificity balanced enough for clinical use?
- Which prediction files should be carried into the paper figures?

## Next notebook

`09_ablation_experiments.ipynb` will compare modality subsets, temporal feature subsets, and fusion strategies to strengthen the research contribution.